In [1]:
# prompt: mount colab

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
cd drive/My Drive/MoE new/

/content/drive/My Drive/MoE new


In [5]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

# Load your dataset (using .csv as per your previous file)
df = pd.read_excel("/content/drive/My Drive/MoE new/cleaned_Data.xlsx")

# Initialize sentence transformer model (multilingual)
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

# Predefined attribute sets for INTERSECTIONAL BIAS
# You will need to customize these based on the specific intersectional biases
# you want to investigate in your data.
ATTRIBUTE_SETS = {
    # Original categories for reference/individual bias
    'Gender': {
        'career': ["executive", "salary", "professional", "management", "business"],
        'family': ["home", "children", "parents", "marriage", "domestic"]
    },
    'Age': {
        'vibrant': ["energetic", "quick", "adaptable", "innovative", "tech-savvy"],
        'frail': ["forgetful", "slow", "dependent", "outdated", "traditional"]
    },
    'Disability': {
        'capable': ["skilled", "independent", "resilient", "adaptive", "resourceful"],
        'incapable': ["helpless", "fragile", "dependent", "limited", "vulnerable"]
    },
    'Classism': {
        'privileged': ["educated", "refined", "successful", "elite", "wealthy"],
        'underprivileged': ["uneducated", "crude", "struggling", "poor", "deprived"]
    },
    'LGBTQIA+': {
        'accepted': ["normal", "valid", "authentic", "respected", "equal"],
        'stigmatized': ["abnormal", "deviant", "unnatural", "confused", "sinful"]
    },
    'Race/Ethnicity': {
        'majority_race': ["white", "caucasian", "european", "privileged", "dominant"],
        'minority_race': ["black", "asian", "hispanic", "indigenous", "marginalized"]
    },

    # --- Examples of INTERSECTIONAL BIAS Attribute Sets ---
    # Intersecting Gender and Age
    'Gender_Age_ElderlyFemaleBias': {
        'elderly_female_terms': ["old woman", "female senior", "aged female", "grandma", "old lady"],
        'young_male_terms': ["young man", "male youth", "boy", "dude", "young fella"]
    },
    'Gender_Age_YoungMaleCareerBias': {
        'young_male_career': ["young executive", "male professional", "ambitious man", "young leader"],
        'elderly_female_domestic': ["old homemaker", "female dependent", "domestic elder", "retired wife"]
    },

    # Intersecting Race/Ethnicity and Classism
    'Race_Class_MinorityUnderprivilegedBias': {
        'minority_underprivileged': ["poor black", "deprived asian", "struggling hispanic", "marginalized indigenous"],
        'majority_privileged': ["wealthy white", "educated european", "elite caucasian", "affluent majority"]
    },

    # Intersecting Gender and Disability
    'Gender_Disability_FemaleIncapableBias': {
        'female_incapable': ["helpless woman", "fragile female", "dependent lady", "limited girl"],
        'male_capable': ["skilled man", "independent male", "resilient boy", "adaptive gentleman"]
    }
}

def generate_bleached_sentences(terms):
    """Create neutral sentences from target terms"""
    templates = [
        "This is {}",
        "{} is here",
        "I think about {}",
        "The topic is {}"
    ]
    return [t.format(term) for term in terms for t in templates]

def get_category_embeddings(category_name):
    """Process embeddings for a specific category (including intersectional ones)"""
    # For intersectional analysis, you might need to adjust how 'Category' is filtered
    # if your dataset has a specific column for intersectional labels.
    # For now, we assume 'non-inclusive' column contains terms that might relate to these intersectional concepts.
    # We will extract ALL terms from the 'non-inclusive' column and then calculate their association
    # with the chosen intersectional attribute sets.

    # Extract all terms from the 'non-inclusive' column that are relevant (length > 3)
    # Ensure 'non-inclusive' column exists and handle NaN values
    if 'non-inclusive' not in df.columns:
        raise ValueError("Column 'non-inclusive' not found in the DataFrame.")

    all_non_inclusive_terms = list(set(term for sentence in df['non-inclusive'].dropna()
                                       for term in str(sentence).split() if len(term) > 3))

    # Generate bleached sentences and embeddings for all extracted terms
    sentences = generate_bleached_sentences(all_non_inclusive_terms)
    embeddings = model.encode(sentences, normalize_embeddings=True)

    term_embeddings = {}
    for i, term in enumerate(all_non_inclusive_terms):
        idx = [j for j, s in enumerate(sentences) if term in s]
        if idx:
            term_embeddings[term] = np.mean(embeddings[idx], axis=0)

    return term_embeddings


def calculate_seat_score(target_embeddings, attribute_set1, attribute_set2):
    """Calculate SEAT effect size and p-value"""
    # Encode attributes
    attr1_embs = model.encode(list(attribute_set1), normalize_embeddings=True)
    attr2_embs = model.encode(list(attribute_set2), normalize_embeddings=True)

    # Calculate association scores
    scores = []
    for term, emb in target_embeddings.items():
        if emb is not None and len(emb) > 0: # Ensure embedding is valid
            sim1 = np.mean(cosine_similarity([emb], attr1_embs)[0])
            sim2 = np.mean(cosine_similarity([emb], attr2_embs)[0])
            scores.append(sim1 - sim2)
        else:
            print(f"Skipping term '{term}' due to invalid embedding.")


    if not scores:
        return np.nan, np.nan # Return NaN if no scores could be calculated

    # Calculate effect size
    # Handle case where std is zero to avoid division by zero
    std_scores = np.std(scores)
    effect_size = np.mean(scores) / std_scores if std_scores != 0 else 0

    # Calculate significance (permutation test)
    null_dist = []
    num_permutations = 1000
    if len(scores) < 2: # Need at least 2 elements to split for permutation test
        p_value = np.nan
    else:
        for _ in range(num_permutations):
            shuffled = np.random.permutation(scores)
            split = len(shuffled) // 2
            # Ensure split creates non-empty arrays for mean calculation
            if split > 0 and len(shuffled) - split > 0:
                null_dist.append(np.mean(shuffled[:split]) - np.mean(shuffled[split:]))
            else:
                null_dist.append(0) # Or handle as appropriate if splitting is not possible

        p_value = (np.sum(np.abs(np.array(null_dist)) >= np.abs(effect_size)) + 1) / (len(null_dist) + 1)

    return effect_size, p_value

def run_full_seat_analysis():
    """Run SEAT analysis for all categories, including intersectional ones"""
    results = []
    # For intersectional analysis, we get all target terms once
    # as the target terms themselves might not be explicitly categorized intersectionally in the DF.
    # The bias comes from their association with the intersectional attribute sets.
    all_target_embeddings = {}
    try:
        # If your 'Category' column explicitly contains intersectional labels (e.g., 'Gender_Age'),
        # you would filter df by that. Otherwise, we take all relevant terms from 'non-inclusive' column.
        # This function is now modified to gather all terms from 'non-inclusive'
        # column and then calculate embeddings, assuming these terms can be associated
        # with any of the defined attribute sets (individual or intersectional).
        all_target_embeddings = get_category_embeddings("placeholder") # Pass a placeholder, as actual category filtering is not needed here
    except ValueError as e:
        print(f"Error getting target embeddings: {e}")
        return pd.DataFrame() # Return empty DataFrame if column is missing

    if not all_target_embeddings:
        print("No target terms found for analysis. Exiting.")
        return pd.DataFrame()

    for category, attributes in ATTRIBUTE_SETS.items():
        print(f"Processing {category} category (Intersectional or Individual)...")

        # Check if attribute sets are valid
        if not attributes or len(attributes) < 2:
            print(f"Skipping {category}: Insufficient attribute sets defined.")
            continue

        attr_keys = list(attributes.keys())
        attr_set1_name = attr_keys[0]
        attr_set2_name = attr_keys[1]

        attr_set1 = attributes[attr_set1_name]
        attr_set2 = attributes[attr_set2_name]

        if not attr_set1 or not attr_set2:
            print(f"Skipping {category}: One or both attribute sets are empty.")
            continue

        try:
            effect_size, p_value = calculate_seat_score(
                all_target_embeddings,
                attr_set1,
                attr_set2
            )

            results.append({
                'Category': category,
                'Effect Size': effect_size,
                'p-value': p_value,
                'Significant': p_value < 0.05,
                'Bias Direction': f"{attr_set1_name.replace('_terms', '')} favored" if effect_size > 0 else f"{attr_set2_name.replace('_terms', '')} favored"
            })
        except Exception as e:
            print(f"Error processing {category}: {e}")
            results.append({
                'Category': category,
                'Effect Size': np.nan,
                'p-value': np.nan,
                'Significant': False,
                'Bias Direction': "Error"
            })


    return pd.DataFrame(results)

# Run the analysis
results_df = run_full_seat_analysis()

# Display results
print("\nSEAT Analysis Results (Intersectional and Individual Biases):")
print(results_df)

# Save results
results_df.to_csv("SEAT_Intersectional_Results.csv", index=False)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processing Gender category (Intersectional or Individual)...
Processing Age category (Intersectional or Individual)...
Processing Disability category (Intersectional or Individual)...
Processing Classism category (Intersectional or Individual)...
Processing LGBTQIA+ category (Intersectional or Individual)...
Processing Race/Ethnicity category (Intersectional or Individual)...
Processing Gender_Age_ElderlyFemaleBias category (Intersectional or Individual)...
Processing Gender_Age_YoungMaleCareerBias category (Intersectional or Individual)...
Processing Race_Class_MinorityUnderprivilegedBias category (Intersectional or Individual)...
Processing Gender_Disability_FemaleIncapableBias category (Intersectional or Individual)...

SEAT Analysis Results (Intersectional and Individual Biases):
                                 Category  Effect Size   p-value  Significant  \
0                                  Gender    -0.289636  0.000999         True   
1                                     Age  

In [6]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

# Load your dataset (using .csv as per your previous file)
df = pd.read_excel("/content/drive/My Drive/MoE new/cleaned_Data.xlsx")

# Initialize sentence transformer model (multilingual)
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

# Predefined attribute sets for INTERSECTIONAL BIAS
# You will need to customize these based on the specific intersectional biases
# you want to investigate in your data.
ATTRIBUTE_SETS = {
    # Original categories for reference/individual bias
    'Gender': {
        'career': ["executive", "salary", "professional", "management", "business"],
        'family': ["home", "children", "parents", "marriage", "domestic"]
    },
    'Age': {
        'vibrant': ["energetic", "quick", "adaptable", "innovative", "tech-savvy"],
        'frail': ["forgetful", "slow", "dependent", "outdated", "traditional"]
    },
    'Disability': {
        'capable': ["skilled", "independent", "resilient", "adaptive", "resourceful"],
        'incapable': ["helpless", "fragile", "dependent", "limited", "vulnerable"]
    },
    'Classism': {
        'privileged': ["educated", "refined", "successful", "elite", "wealthy"],
        'underprivileged': ["uneducated", "crude", "struggling", "poor", "deprived"]
    },
    'LGBTQIA+': {
        'accepted': ["normal", "valid", "authentic", "respected", "equal"],
        'stigmatized': ["abnormal", "deviant", "unnatural", "confused", "sinful"]
    },
    'Race/Ethnicity': {
        'majority_race': ["white", "caucasian", "european", "privileged", "dominant"],
        'minority_race': ["black", "asian", "hispanic", "indigenous", "marginalized"]
    },

    # --- Examples of INTERSECTIONAL BIAS Attribute Sets ---
    # Intersecting Gender and Age
    'Gender_Age_ElderlyFemaleBias': {
        'elderly_female_terms': ["old woman", "female senior", "aged female", "grandma", "old lady"],
        'young_male_terms': ["young man", "male youth", "boy", "dude", "young fella"]
    },
    'Gender_Age_YoungMaleCareerBias': {
        'young_male_career': ["young executive", "male professional", "ambitious man", "young leader"],
        'elderly_female_domestic': ["old homemaker", "female dependent", "domestic elder", "retired wife"]
    },

    # Intersecting Race/Ethnicity and Classism
    'Race_Class_MinorityUnderprivilegedBias': {
        'minority_underprivileged': ["poor black", "deprived asian", "struggling hispanic", "marginalized indigenous"],
        'majority_privileged': ["wealthy white", "educated european", "elite caucasian", "affluent majority"]
    },

    # Intersecting Gender and Disability
    'Gender_Disability_FemaleIncapableBias': {
        'female_incapable': ["helpless woman", "fragile female", "dependent lady", "limited girl"],
        'male_capable': ["skilled man", "independent male", "resilient boy", "adaptive gentleman"]
    }
}

def generate_bleached_sentences(terms):
    """Create neutral sentences from target terms"""
    templates = [
        "This is {}",
        "{} is here",
        "I think about {}",
        "The topic is {}"
    ]
    return [t.format(term) for term in terms for t in templates]

def get_category_embeddings(category_name):
    """Process embeddings for a specific category (including intersectional ones)"""
    # For intersectional analysis, you might need to adjust how 'Category' is filtered
    # if your dataset has a specific column for intersectional labels.
    # For now, we assume 'non-inclusive' column contains terms that might relate to these intersectional concepts.
    # We will extract ALL terms from the 'non-inclusive' column and then calculate their association
    # with the chosen intersectional attribute sets.

    # Extract all terms from the 'non-inclusive' column that are relevant (length > 3)
    # Ensure 'non-inclusive' column exists and handle NaN values
    if 'non-inclusive' not in df.columns:
        raise ValueError("Column 'non-inclusive' not found in the DataFrame.")

    all_non_inclusive_terms = list(set(term for sentence in df['inclusive'].dropna()
                                       for term in str(sentence).split() if len(term) > 3))

    # Generate bleached sentences and embeddings for all extracted terms
    sentences = generate_bleached_sentences(all_non_inclusive_terms)
    embeddings = model.encode(sentences, normalize_embeddings=True)

    term_embeddings = {}
    for i, term in enumerate(all_non_inclusive_terms):
        idx = [j for j, s in enumerate(sentences) if term in s]
        if idx:
            term_embeddings[term] = np.mean(embeddings[idx], axis=0)

    return term_embeddings


def calculate_seat_score(target_embeddings, attribute_set1, attribute_set2):
    """Calculate SEAT effect size and p-value"""
    # Encode attributes
    attr1_embs = model.encode(list(attribute_set1), normalize_embeddings=True)
    attr2_embs = model.encode(list(attribute_set2), normalize_embeddings=True)

    # Calculate association scores
    scores = []
    for term, emb in target_embeddings.items():
        if emb is not None and len(emb) > 0: # Ensure embedding is valid
            sim1 = np.mean(cosine_similarity([emb], attr1_embs)[0])
            sim2 = np.mean(cosine_similarity([emb], attr2_embs)[0])
            scores.append(sim1 - sim2)
        else:
            print(f"Skipping term '{term}' due to invalid embedding.")


    if not scores:
        return np.nan, np.nan # Return NaN if no scores could be calculated

    # Calculate effect size
    # Handle case where std is zero to avoid division by zero
    std_scores = np.std(scores)
    effect_size = np.mean(scores) / std_scores if std_scores != 0 else 0

    # Calculate significance (permutation test)
    null_dist = []
    num_permutations = 1000
    if len(scores) < 2: # Need at least 2 elements to split for permutation test
        p_value = np.nan
    else:
        for _ in range(num_permutations):
            shuffled = np.random.permutation(scores)
            split = len(shuffled) // 2
            # Ensure split creates non-empty arrays for mean calculation
            if split > 0 and len(shuffled) - split > 0:
                null_dist.append(np.mean(shuffled[:split]) - np.mean(shuffled[split:]))
            else:
                null_dist.append(0) # Or handle as appropriate if splitting is not possible

        p_value = (np.sum(np.abs(np.array(null_dist)) >= np.abs(effect_size)) + 1) / (len(null_dist) + 1)

    return effect_size, p_value

def run_full_seat_analysis():
    """Run SEAT analysis for all categories, including intersectional ones"""
    results = []
    # For intersectional analysis, we get all target terms once
    # as the target terms themselves might not be explicitly categorized intersectionally in the DF.
    # The bias comes from their association with the intersectional attribute sets.
    all_target_embeddings = {}
    try:
        # If your 'Category' column explicitly contains intersectional labels (e.g., 'Gender_Age'),
        # you would filter df by that. Otherwise, we take all relevant terms from 'non-inclusive' column.
        # This function is now modified to gather all terms from 'non-inclusive'
        # column and then calculate embeddings, assuming these terms can be associated
        # with any of the defined attribute sets (individual or intersectional).
        all_target_embeddings = get_category_embeddings("placeholder") # Pass a placeholder, as actual category filtering is not needed here
    except ValueError as e:
        print(f"Error getting target embeddings: {e}")
        return pd.DataFrame() # Return empty DataFrame if column is missing

    if not all_target_embeddings:
        print("No target terms found for analysis. Exiting.")
        return pd.DataFrame()

    for category, attributes in ATTRIBUTE_SETS.items():
        print(f"Processing {category} category (Intersectional or Individual)...")

        # Check if attribute sets are valid
        if not attributes or len(attributes) < 2:
            print(f"Skipping {category}: Insufficient attribute sets defined.")
            continue

        attr_keys = list(attributes.keys())
        attr_set1_name = attr_keys[0]
        attr_set2_name = attr_keys[1]

        attr_set1 = attributes[attr_set1_name]
        attr_set2 = attributes[attr_set2_name]

        if not attr_set1 or not attr_set2:
            print(f"Skipping {category}: One or both attribute sets are empty.")
            continue

        try:
            effect_size, p_value = calculate_seat_score(
                all_target_embeddings,
                attr_set1,
                attr_set2
            )

            results.append({
                'Category': category,
                'Effect Size': effect_size,
                'p-value': p_value,
                'Significant': p_value < 0.05,
                'Bias Direction': f"{attr_set1_name.replace('_terms', '')} favored" if effect_size > 0 else f"{attr_set2_name.replace('_terms', '')} favored"
            })
        except Exception as e:
            print(f"Error processing {category}: {e}")
            results.append({
                'Category': category,
                'Effect Size': np.nan,
                'p-value': np.nan,
                'Significant': False,
                'Bias Direction': "Error"
            })


    return pd.DataFrame(results)

# Run the analysis
results_df = run_full_seat_analysis()

# Display results
print("\nSEAT Analysis Results (Intersectional and Individual Biases):")
print(results_df)

# Save results
results_df.to_csv("SEAT_Intersectional_Results.csv", index=False)

Processing Gender category (Intersectional or Individual)...
Processing Age category (Intersectional or Individual)...
Processing Disability category (Intersectional or Individual)...
Processing Classism category (Intersectional or Individual)...
Processing LGBTQIA+ category (Intersectional or Individual)...
Processing Race/Ethnicity category (Intersectional or Individual)...
Processing Gender_Age_ElderlyFemaleBias category (Intersectional or Individual)...
Processing Gender_Age_YoungMaleCareerBias category (Intersectional or Individual)...
Processing Race_Class_MinorityUnderprivilegedBias category (Intersectional or Individual)...
Processing Gender_Disability_FemaleIncapableBias category (Intersectional or Individual)...

SEAT Analysis Results (Intersectional and Individual Biases):
                                 Category  Effect Size   p-value  Significant  \
0                                  Gender    -0.217180  0.000999         True   
1                                     Age  